# PRACTICAL 5: Feature Engineering for Anomaly Detection

## Objective
Extract 5 quantitative numerical features from `labeled_logs.csv` to prepare the security log data for machine learning model training:
1. `ip_request_count`: Total requests per IP.
2. `time_diff_seconds`: Time difference between consecutive requests from the same IP.
3. `status_code_404_count`: Count of 404 HTTP errors by IP address.
4. `is_bot_agent`: Binary flag for security scanners / automated bot tools.
5. `url_entropy`: Shannon Entropy of requested resource URLs.

In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

plt.style.use('seaborn-v0_8')
%matplotlib inline

### 1. Load Labeled Log Data

In [ ]:
input_file = "labeled_logs.csv"
print(f"Loading dataset from '{input_file}'...")
df = pd.read_csv(input_file, low_memory=False)
print(f"Total Records Loaded: {len(df):,}")

### 2. Feature Extraction

In [ ]:
def calculate_entropy(text):
    if not text or pd.isna(text):
        return 0.0
    text_str = str(text)
    prob = [float(text_str.count(c)) / len(text_str) for c in set(text_str)]
    return -sum([p * math.log2(p) for p in prob])

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.sort_values(by=['ip', 'timestamp']).reset_index(drop=True)

# 1. IP Request Count
df['ip_request_count'] = df['ip'].map(df['ip'].value_counts())

# 2. Time gap
df['time_diff_seconds'] = df.groupby('ip')['timestamp'].diff().dt.total_seconds().fillna(0.0)

# 3. 404 Status Code Count
df['is_404'] = (df['status_code'] == 404).astype(int)
df['status_code_404_count'] = df.groupby('ip')['is_404'].transform('sum')
df.drop(columns=['is_404'], inplace=True)

# 4. Bot Agent Flag
bot_keywords = r"(?:bot|crawler|spider|scraper|sqlmap|nikto|gobuster|dirbuster|curl|python-urllib|nmap)"
df['is_bot_agent'] = df['user_agent'].astype(str).str.contains(bot_keywords, regex=True, case=False).astype(int)

# 5. URL Entropy
unique_urls = df['resource_requested'].astype(str).unique()
entropy_dict = {url: calculate_entropy(url) for url in unique_urls}
df['url_entropy'] = df['resource_requested'].astype(str).map(entropy_dict)

### 3. Display Feature Summary & Save Visualizations

In [ ]:
feature_cols = ['ip_request_count', 'time_diff_seconds', 'status_code_404_count', 'is_bot_agent', 'url_entropy']
print(df[feature_cols].describe())

os.makedirs("../plots", exist_ok=True)
plt.figure(figsize=(12, 8))
for i, col in enumerate(feature_cols, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], bins=30, kde=True, color='#1f77b4', edgecolor='black')
    plt.title(f"Distribution: {col}", fontsize=11, fontweight='bold')
    plt.yscale('log')
    plt.tight_layout()

plt.savefig("../plots/plot_p5_feature_distributions.png", dpi=300)
plt.show()

output_file = "engineered_features.csv"
df.to_csv(output_file, index=False)
print(f"Saved dataset to '{output_file}'!")

### 5b. Tool attribution carried forward
bot_family is metadata (not an ML feature), so Practicals 6–9 are unaffected.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from bot_taxonomy import classify_bot

if "bot_family" not in df.columns:
    df["bot_family"] = df["user_agent"].astype(str).map(classify_bot)
print("bot_family values:", df["bot_family"].unique().tolist())
print("ML feature columns (unchanged):", ["ip_request_count", "time_diff_seconds", "status_code_404_count", "is_bot_agent", "url_entropy"])

### 5c. SQLi variant carried forward
sqli_variant is metadata (not an ML feature), so Practicals 6–9 are unaffected.

In [ ]:
from sqli_taxonomy import classify_sqli

if "sqli_variant" not in df.columns:
    df["sqli_variant"] = "-"
    is_sqli = df["label"].astype(str).str.lower() == "sqli"
    df.loc[is_sqli, "sqli_variant"] = df.loc[is_sqli, "resource_requested"].astype(str).map(classify_sqli)
print("sqli_variant values:", df["sqli_variant"].unique().tolist())